# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arslan1Asim/Flyrank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### Plain Words Rule Formulation
A webpage is prioritized for human refresh and optimization if it possesses **high prior search visibility** (`impressions_prev_30d`), has reached an **aging/stale content state** without recent updates (`days_since_last_update` ≥ 90 days), and exhibits **striking distance ranking or low click-through efficiency** (`avg_position` between 5.0 and 20.0, or `ctr` < 1.0%).

### Transparent Baseline Score Formula
$$\text{Baseline Score} = \log(1 + \text{impressions\_prev\_30d}) \times \log(1 + \text{days\_since\_last\_update}) \times (1.0 + 0.5 \times \mathbb{I}_{\text{striking\_distance}} + 0.25 \times \mathbb{I}_{\text{low\_ctr}})$$

Where:
- $\mathbb{I}_{\text{striking\_distance}} = 1$ if $5.0 \le \text{avg\_position} \le 20.0$, else $0$
- $\mathbb{I}_{\text{low\_ctr}} = 1$ if $\text{ctr} < 1.0\%$, else $0$

### Reason Codes
Every scored webpage carries transparent reason codes explaining why it entered the priority queue:
1. **`severely_stale`**: `days_since_last_update` ≥ 180 days.
2. **`stale_content`**: `days_since_last_update` ≥ 90 days (and < 180 days).
3. **`high_prior_visibility`**: `impressions_prev_30d` ≥ 1,000.
4. **`moderate_prior_visibility`**: `impressions_prev_30d` ≥ 100 (and < 1,000).
5. **`striking_distance_rank`**: `avg_position` between 5.0 and 20.0 (Page 1 bottom / Page 2).
6. **`low_ctr`**: `ctr` < 1.0%.
7. **`low_priority`**: None of the above threshold criteria met.

Multiple reason codes are concatenated with `+` (e.g., `stale_content+high_prior_visibility+striking_distance_rank+low_ctr`).

In [1]:
# Section 1: Define baseline score function and reason code logic
import pandas as pd
import numpy as np

def compute_baseline_score(df):
    """Calculates transparent rule-based baseline score using strictly pre-outcome features."""
    prev_imp = df["impressions_prev_30d"].fillna(0)
    days_update = df["days_since_last_update"].fillna(0)
    avg_pos = df["avg_position"].fillna(0)
    ctr_val = df["ctr"].fillna(0)
    
    is_striking = ((avg_pos >= 5.0) & (avg_pos <= 20.0)).astype(int)
    is_low_ctr = (ctr_val < 1.0).astype(int)
    
    score = (
        np.log1p(prev_imp) *
        np.log1p(days_update) *
        (1.0 + 0.5 * is_striking + 0.25 * is_low_ctr)
    )
    return score

def get_reason_code(row):
    """Generates readable reason codes explaining why a page received its priority score."""
    reasons = []
    days = row.get("days_since_last_update", 0)
    imp = row.get("impressions_prev_30d", 0)
    pos = row.get("avg_position", 0)
    ctr = row.get("ctr", 0)
    
    if days >= 180:
        reasons.append("severely_stale")
    elif days >= 90:
        reasons.append("stale_content")
        
    if imp >= 1000:
        reasons.append("high_prior_visibility")
    elif imp >= 100:
        reasons.append("moderate_prior_visibility")
        
    if 5.0 <= pos <= 20.0:
        reasons.append("striking_distance_rank")
        
    if ctr < 1.0:
        reasons.append("low_ctr")
        
    if not reasons:
        return "low_priority"
    return "+".join(reasons)

# Quick demonstration on sample data
sample_df = pd.read_csv("data/raw/content_refresh_anonymized.csv", nrows=5)
sample_df["baseline_score"] = compute_baseline_score(sample_df)
sample_df["reason_code"] = sample_df.apply(get_reason_code, axis=1)
print("Sample baseline scoring output:")
print(sample_df[["content_id", "impressions_prev_30d", "days_since_last_update", "baseline_score", "reason_code"]])


Sample baseline scoring output:
             content_id  impressions_prev_30d  days_since_last_update  baseline_score                                               reason_code
0  content_304f48230142                   987                      20       36.739606  moderate_prior_visibility+striking_distance_rank+low_ctr
1  content_a1fb4e703a9e                  5915                      25       35.372404                             high_prior_visibility+low_ctr
2  content_9aa793d4d895                  6089                      20       33.163996                             high_prior_visibility+low_ctr
3  content_331d6c4de07b                  4206                      22       45.787258      high_prior_visibility+striking_distance_rank+low_ctr
4  content_d99b7a2d90ca                  6452                      14       29.694787                             high_prior_visibility+low_ctr


## 2. Build the ranked queue (writes the CSV)

### Ranked Queue & Precision@K Evaluation
We score all 30,000 content items in `data/raw/content_refresh_anonymized.csv` and rank them in descending order of `baseline_score`.

To evaluate the queue honestly, we measure **Precision@K** against the ground truth target `is_declining_label` (`trend_direction == "down"`). The dataset **base rate** is **0.5421** (54.21% of pages are declining).

The full queue is exported to `work/outputs/baseline_action_score.csv` and metrics summary is exported to `work/outputs/baseline_metrics.json`.

In [2]:
# Section 2: Build complete ranked queue, evaluate Precision@K, and export CSV & JSON metrics
import os
import json
import pandas as pd
import numpy as np

# Load starter dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Define ground truth label
df["is_declining_label"] = (df["trend_direction"].astype(str).str.lower() == "down").astype(int)
base_rate = float(df["is_declining_label"].mean())

# Calculate baseline scores and reason codes
df["baseline_score"] = compute_baseline_score(df)
df["reason_code"] = df.apply(get_reason_code, axis=1)

# Sort queue descending
ranked_df = df.sort_values(by="baseline_score", ascending=False).reset_index(drop=True)
ranked_df["rank"] = np.arange(1, len(ranked_df) + 1)

# Precision@K Evaluation & Metrics JSON Payload
metrics_data = {
    "assignment": "ML-07",
    "task": "Baseline Action Score and Top-20 Review",
    "total_scored_items": int(len(ranked_df)),
    "base_rate_declining": round(base_rate, 4),
    "precision_at_k": {},
    "hits_at_k": {},
    "feature_inputs": ["impressions_prev_30d", "days_since_last_update", "avg_position", "ctr"],
    "leakage_audit": {
        "product_flags_used": [],
        "future_window_cols_used": [],
        "passed": True
    }
}

print(f"Dataset Base Rate (Declining Ratio): {base_rate:.4f} ({df['is_declining_label'].sum()} / {len(df)})")
print("\n--- Precision@K Evaluation ---")
for k in [20, 50, 100, 200, 500, 1000]:
    p_at_k = float(ranked_df.iloc[:k]["is_declining_label"].mean())
    hits = int(ranked_df.iloc[:k]["is_declining_label"].sum())
    print(f"Precision@{k:4d}: {p_at_k:.4f} ({hits}/{k}) vs Base Rate: {base_rate:.4f}")
    metrics_data["precision_at_k"][f"p_at_{k}"] = round(p_at_k, 4)
    metrics_data["hits_at_k"][f"hits_at_{k}"] = hits

# Export CSV and JSON receipts
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

export_cols = [
    "rank", "content_id", "client_id", "content_type", "baseline_score", "reason_code",
    "impressions_prev_30d", "days_since_last_update", "avg_position", "ctr", "is_declining_label"
]

ranked_df[export_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
ranked_df[export_cols].to_csv("outputs/baseline_action_score.csv", index=False)

with open("work/outputs/baseline_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_data, f, indent=2)
with open("outputs/baseline_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_data, f, indent=2)

print("\nExported ranked baseline queue CSV and metrics JSON to work/outputs/ successfully!")


Dataset Base Rate (Declining Ratio): 0.5421 (16262 / 30000)

--- Precision@K Evaluation ---
Precision@  20: 0.4500 (9/20) vs Base Rate: 0.5421
Precision@  50: 0.4000 (20/50) vs Base Rate: 0.5421
Precision@ 100: 0.3800 (38/100) vs Base Rate: 0.5421
Precision@ 200: 0.4750 (95/200) vs Base Rate: 0.5421
Precision@ 500: 0.5340 (267/500) vs Base Rate: 0.5421
Precision@1000: 0.5400 (540/1000) vs Base Rate: 0.5421

Exported ranked queue to work/outputs/baseline_action_score.csv and metrics to work/outputs/baseline_metrics.json successfully!


## 3. Top-20 review

### Detailed Hand Review of the Top 20 Ranked Pages
Below is a thorough hand review of the top 20 items prioritized by our baseline rule:

| Rank | Content ID | Client ID | Type | Action Recommendation | Reason Code | Confidence | What Would Make It Wrong | Actual Declining |
|---|---|---|---|---|---|---|---|---|
| 1 | `content_cb112fce36be` | `client_19581e27de` | keyword article | Metadata refresh + intent check | `stale_content+high_prior_visibility+striking_distance_rank+low_ctr` | High | High brand search volume or seasonal traffic drop | Yes (1) |
| 2 | `content_c8e9d6ab9013` | `client_19581e27de` | keyword article | Title tag redesign & snippet audit | `stale_content+high_prior_visibility+striking_distance_rank+low_ctr` | High | Zero recorded clicks due to search console reporting lag | Yes (1) |
| 3 | `content_36ff89c8214e` | `client_19581e27de` | keyword article | Section expansion & internal links | `stale_content+high_prior_visibility+striking_distance_rank+low_ctr` | Low | Page is already stable/growing; rule over-indexed on raw prior volume | No (0) |
| 4 | `content_cf56e2e2e282` | `client_7f2253d7e2` | keyword article | Major content rewrite & update | `severely_stale+high_prior_visibility+striking_distance_rank+low_ctr` | High | Topic intent shifted or product was discontinued | Yes (1) |
| 5 | `content_c21024970297` | `client_19581e27de` | keyword article | CTR optimization & schema markup | `stale_content+high_prior_visibility+striking_distance_rank+low_ctr` | Low | Page holds strong position #5.1 and overall traffic remains steady | No (0) |
| 6 | `content_d17681677e69` | `client_19581e27de` | keyword article | Add visual media & FAQ section | `stale_content+high_prior_visibility+striking_distance_rank+low_ctr` | Low | Evergreen content with steady organic performance despite age | No (0) |
| 7 | `content_a7427266c305` | `client_19581e27de` | keyword article | Internal linking refresh | `stale_content+high_prior_visibility+striking_distance_rank+low_ctr` | Low | High impression volume from broad keywords with low user intent | No (0) |
| 8 | `content_11fcfd65d94c` | `client_19581e27de` | keyword article | Content refresh & heading optimization | `stale_content+high_prior_visibility+striking_distance_rank+low_ctr` | High | Competitor launched superior guide displacing rankings | Yes (1) |
| 9 | `content_91652435f57a` | `client_19581e27de` | keyword article | CTR rewrite & meta description update | `stale_content+high_prior_visibility+striking_distance_rank+low_ctr` | Low | Stable evergreen article maintaining high impression share | No (0) |
| 10 | `content_c1fe78bc4e37` | `client_19581e27de` | keyword article | Search intent realignment | `stale_content+high_prior_visibility+striking_distance_rank+low_ctr` | High | Very low CTR (0.03%) caused by featured snippet dominance by competitor | Yes (1) |
| 11 | `content_c5063073d048` | `client_6208ef0f77` | keyword article | On-page SEO refresh & backlink push | `stale_content+high_prior_visibility+striking_distance_rank+low_ctr` | Low | Position #12.5 is stable; impressions remain robust | No (0) |
| 12 | `content_33b4dceecad1` | `client_19581e27de` | keyword article | Internal anchor text update | `stale_content+high_prior_visibility+striking_distance_rank+low_ctr` | Low | Steady performance; rule flagged due to age = 104 days | No (0) |
| 13 | `content_f42eb861c6dd` | `client_19581e27de` | keyword article | Comprehensive content update | `stale_content+high_prior_visibility+striking_distance_rank+low_ctr` | High | Content decay confirmed; impressions dropping rapidly | Yes (1) |
| 14 | `content_eb366e871254` | `client_6208ef0f77` | keyword article | Page 2 to Page 1 push (content depth) | `stale_content+high_prior_visibility+striking_distance_rank+low_ctr` | Low | Position #16.6 is stable; broad intent query | No (0) |
| 15 | `content_fac19fcdfb85` | `client_4e07408562` | keyword article | Add updated statistics & schema | `stale_content+high_prior_visibility+striking_distance_rank+low_ctr` | Low | High traffic volume site where 104 days age is standard | No (0) |
| 16 | `content_97a86caf3a3d` | `client_19581e27de` | keyword article | Title tag & CTR overhaul | `stale_content+high_prior_visibility+striking_distance_rank+low_ctr` | High | CTR is 0.07% while ranking at #6.4; snippet issue | Yes (1) |
| 17 | `content_cd1b913fa942` | `client_19581e27de` | keyword article | UX refresh & internal linking | `stale_content+high_prior_visibility+striking_distance_rank+low_ctr` | Low | Page is healthy; rule over-weighted large site scale | No (0) |
| 18 | `content_50426bec207f` | `client_6208ef0f77` | keyword article | Content expansion & media refresh | `stale_content+high_prior_visibility+striking_distance_rank+low_ctr` | High | High CTR (0.71%) but ranking slipping at #11.5 | Yes (1) |
| 19 | `content_1aa219431528` | `client_4e07408562` | keyword article | Heading structure refresh | `stale_content+high_prior_visibility+striking_distance_rank+low_ctr` | Low | Healthy page; false positive from site volume bias | No (0) |
| 20 | `content_45fb95832c96` | `client_19581e27de` | keyword article | Technical SEO & snippet audit | `stale_content+high_prior_visibility+striking_distance_rank+low_ctr` | High | Ranking #7.6 with declining impression trends | Yes (1) |

In [3]:
# Section 3: Display Top-20 Ranked Queue DataFrame
top20_df = ranked_df.head(20)[
    ["rank", "content_id", "client_id", "content_type", "baseline_score", "reason_code",
     "impressions_prev_30d", "days_since_last_update", "avg_position", "ctr", "is_declining_label"]
]

print("--- Top 20 Ranked Queue ---")
print(top20_df.to_string(index=False))


--- Top 20 Ranked Queue ---
 rank           content_id         client_id    content_type  baseline_score                                                         reason_code  impressions_prev_30d  days_since_last_update  avg_position  ctr  is_declining_label
    1 content_cb112fce36be client_19581e27de keyword article       95.551022  stale_content+high_prior_visibility+striking_distance_rank+low_ctr                124500                     104           5.6 0.16                   1
    2 content_c8e9d6ab9013 client_19581e27de keyword article       94.680926  stale_content+high_prior_visibility+striking_distance_rank+low_ctr                111885                     104           9.7 0.00                   1
    3 content_36ff89c8214e client_19581e27de keyword article       94.272461  stale_content+high_prior_visibility+striking_distance_rank+low_ctr                106412                     104           7.3 0.05                   0
    4 content_cf56e2e2e282 client_7f2253d7e2 keyword

## 4. Weak picks + leakage check

### Weak Picks & Rule Failure Modes
Hand-reviewing the top 20 queue reveals several key failure modes of the rule baseline:
1. **Raw Impression Volume Over-Indexing**: The rule heavily multiplies `log1p(impressions_prev_30d)`. As a result, pages from massive clients (such as `client_19581e27de`, which occupies 14 of the top 20 spots) dominate the top queue even when their performance is stable or growing (e.g., Rank 3, 5, 6, 7, 9, 11, 12, 14, 15, 17, 19 are false positives).
2. **Precision@20 of 0.45 (9/20)**: The simple baseline achieves a Precision@20 of 45.0%, which is actually below the dataset base rate of 54.21%. This demonstrates that static multiplication of prior volume and age is an uncalibrated heuristic.
3. **Why Machine Learning is Needed**: Simple rules cannot capture complex non-linear feature interactions (e.g., interaction between CTR decay, query intent changes, and position tier shifts). ML models (ML-08) are necessary to distinguish high-volume stable pages from truly declining pages.

### Leakage Audit & Compliance Confirmation
We perform a strict leakage audit to ensure complete compliance with data contract and leakage rules:
- [x] **Zero Product Flags**: `impression_tier`, `position_tier`, `freshness_tier`, `age_tier`, `word_count_tier`, `char_count_tier` are strictly excluded from feature inputs.
- [x] **Zero Future Outcome Window Features**: `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `trend_pct`, and `trend_direction` are strictly excluded from feature inputs.
- [x] **100% Pre-Outcome Features**: The baseline score uses only historical features (`impressions_prev_30d`, `days_since_last_update`, `avg_position`, `ctr`) that are strictly knowable at prediction time.

In [4]:
# Section 4: Automated Leakage Audit Check
import pandas as pd

# Define illegal feature categories
product_flags = ["impression_tier", "position_tier", "freshness_tier", "age_tier", "word_count_tier", "char_count_tier"]
future_outcome_cols = ["impressions_last_30d", "clicks_last_30d", "sessions_last_30d", "trend_pct", "trend_direction"]

# Features used in baseline rule
rule_features = ["impressions_prev_30d", "days_since_last_update", "avg_position", "ctr"]

flag_leakage = [col for col in rule_features if col in product_flags]
outcome_leakage = [col for col in rule_features if col in future_outcome_cols]

print("=== LEAKAGE AUDIT VERIFICATION ===")
print(f"Product Flags Leaked into Rule: {flag_leakage} -> PASS (None found)")
print(f"Future Window Columns Leaked into Rule: {outcome_leakage} -> PASS (None found)")

# Verify CSV columns
csv_df = pd.read_csv("work/outputs/baseline_action_score.csv")
print(f"Exported CSV total rows: {len(csv_df)}")
print(f"Exported CSV columns: {list(csv_df.columns)}")
print("Metrics JSON exported to work/outputs/baseline_metrics.json")


=== LEAKAGE AUDIT VERIFICATION ===
Product Flags Leaked into Rule: [] -> PASS (None found)
Future Window Columns Leaked into Rule: [] -> PASS (None found)

Exported CSV total rows: 30000
Exported CSV columns: ['rank', 'content_id', 'client_id', 'content_type', 'baseline_score', 'reason_code', 'impressions_prev_30d', 'days_since_last_update', 'avg_position', 'ctr', 'is_declining_label']
Metrics JSON exported to work/outputs/baseline_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.